## Setup vLLM and LMCache server
Follow below instructions (1-4), then proceed to below Python cell.
1. Start LMCache server first
```sh
lmcache server \
    --l1-size-gb 8 \
    --eviction-policy LRU \
    --chunk-size 256 \
    --port 6555 \
    --http-port 8080 \
    --shm-name lmcache_kvcache_sdk_e2e \
    --no-l1-use-lazy
```
2. Wait until LMCache server is ready
```sh
curl -sf http://localhost:8080/healthcheck && echo " LMCache ready"
```
3. Start vLLM once LMCache server is ready
```sh
env -u VLLM_PORT \
    CUDA_VISIBLE_DEVICES=0 \
    VLLM_ENABLE_V1_MULTIPROCESSING=0 \
    VLLM_BATCH_INVARIANT=1 \
    PYTHONHASHSEED=0 \
    vllm serve Qwen/Qwen3-8B \
    --port 8000 \
    --served-model-name Qwen/Qwen3-8B \
    --no-enable-prefix-caching \
    --enforce-eager \
    --max-model-len 4096 \
    --gpu-memory-utilization 0.6 \
    --kv-transfer-config '{"kv_connector":"LMCacheMPConnector","kv_role":"kv_both","kv_load_failure_policy":"recompute","kv_connector_extra_config":{"lmcache.mp.host":"tcp://localhost","lmcache.mp.port":6555,"lmcache.mp.mq_timeout":10}}' \
    --override-generation-config '{"temperature": 0}' \
    --trust-remote-code
```
4. Wait until vLLM is ready
```sh
curl -sf http://localhost:8000/v1/models && echo " vLLM ready"
```

## Run E2E KV Edit Example

In [1]:
# SPDX-License-Identifier: Apache-2.0
"""End-to-end KV cache remapping driver for the SDK example."""

# Standard
from dataclasses import dataclass
from itertools import islice
import json
import time
from typing import cast

# Third Party
import httpx
from transformers import AutoTokenizer, PreTrainedTokenizerBase

# First Party
import lmcache.sdk.kvcache as lmc_sdk

/home/rani/LMCache/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[2026-06-23 23:26:41,202] LMCache INFO:  torch_dev=<module 'torch.cuda' from '/home/rani/LMCache/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py'>, torch_device_type=cuda (__init__.py:62:lmcache)
[2026-06-23 23:26:41,267] LMCache INFO: Skipping backend lmcache.v1.platform.musa.ops: predicate returned False (__init__.py:102:lmcache)
[2026-06-23 23:26:41,268] LMCache INFO: Skipping backend lmcache.xpu_ops: predicate returned False (__init__.py:102:lmcache)
[2026-06-23 23:26:41,269] LMCache INFO: Using backend: lmcache.c_ops (__init__.py:120:lmcache)
[2026-06-23 23:26:41,604] LMCache INFO: multi_layer_block_kv_transfer mode: ptr (base.py:94:lmcache.v1.multiprocess.transfer_context.base)


In [2]:
model_name = "Qwen/Qwen3-8B"
vllm_url = "http://localhost:8000"
lmcache_url = "http://localhost:8080"
lmcache_mq_url = "tcp://localhost:6555"
chunk_size = 256
min_prompt_tokens = chunk_size * 2
fake_prefix_tokens = 32
max_tokens = 32
timeout = 60  # seconds
trust_remote_code = True

In [3]:
@dataclass(frozen=True)
class CompletionResult:
    """Text and latency returned by one OpenAI-compatible completion call."""

    text: str
    elapsed_seconds: float


def _post_completion(
    *,
    vllm_url: str,
    model_name: str,
    prompt: str | list[int],
    max_tokens: int,
    timeout: float,
) -> CompletionResult:
    """Send one non-streaming completion request to vLLM."""
    payload = {
        "model": model_name,
        "prompt": prompt,
        "max_tokens": max_tokens,
        "min_tokens": max_tokens,
        "temperature": 0,
        "seed": 0,
        "ignore_eos": True,
    }
    start = time.perf_counter()
    response = httpx.post(
        f"{vllm_url.rstrip('/')}/v1/completions",
        json=payload,
        timeout=timeout,
    )
    elapsed = time.perf_counter() - start
    response.raise_for_status()
    body = response.json()
    choices = body.get("choices")
    if not isinstance(choices, list) or not choices:
        raise RuntimeError(f"completion response missing choices: {body}")
    first_choice = choices[0]
    if not isinstance(first_choice, dict) or not isinstance(
        first_choice.get("text"), str
    ):
        raise RuntimeError(f"completion response has invalid choice: {body}")
    return CompletionResult(text=first_choice["text"], elapsed_seconds=elapsed)

In [ ]:
SOURCE_PARAGRAPH = (
    "A systems researcher is studying how an inference cache changes the "
    "latency profile of a long language model prompt. The notes discuss "
    "attention keys, attention values, memory tiers, token chunks, and the "
    "careful measurement of cold and warm requests."
)


def _build_prompts(
    tokenizer: PreTrainedTokenizerBase,
    *,
    min_prompt_tokens: int,
    chunk_size: int,
    fake_prefix_tokens: int,
) -> tuple[list[int], list[int], int]:
    """Build equal-length source/target prompts and the cached-prefix length.
    The prompts differ only in ``fake_prefix_tokens`` synthetic leading token IDs."""
    if not 0 < fake_prefix_tokens < chunk_size:
        raise ValueError(
            f"fake_prefix_tokens must be in (0, {chunk_size}), got {fake_prefix_tokens}"
        )
    min_cache = max(1, min_prompt_tokens - fake_prefix_tokens)
    cache_tokens = ((min_cache + chunk_size - 1) // chunk_size) * chunk_size

    # Two distinct, non-special lead IDs for the source/target prefixes.
    special = {int(t) for t in tokenizer.all_special_ids}
    candidates = (t for t in range(1000, int(tokenizer.vocab_size)) if t not in special)
    leads = list(islice(candidates, 2))
    if len(leads) < 2:
        raise ValueError("could not find two usable non-special token IDs")
    source_lead, target_lead = leads

    # Repeat the paragraph until it covers the suffix, then take the trailing slice.
    suffix_len = cache_tokens - fake_prefix_tokens
    text = SOURCE_PARAGRAPH
    suffix = tokenizer.encode(text, add_special_tokens=False)
    while len(suffix) < suffix_len:
        text = f"{text}\n\n{SOURCE_PARAGRAPH}"
        suffix = tokenizer.encode(text, add_special_tokens=False)
    suffix = [int(t) for t in suffix[-suffix_len:]]

    source = [source_lead] * fake_prefix_tokens + suffix
    target = [target_lead] * fake_prefix_tokens + suffix
    return source, target, cache_tokens

In [5]:
# Only create one LMCacheKVCacheContext, reuse for retrieve() and store()
# multiple times. Only close() once at the end.
ctx = lmc_sdk.connect(
    url=lmcache_mq_url,
    http_url=lmcache_url,
    model_name=model_name,
    timeout=timeout,
)

[2026-06-23 23:26:52,200] LMCache INFO: Initialized LMCacheKVCacheContext with instance_id=3403760, model_name=Qwen/Qwen3-8B, chunk_size=256, shm_name=lmcache_l1_pool_lmcache_kvcache_sdk_e2e (kvcache.py:104:lmcache.sdk.kvcache)
[2026-06-23 23:26:52,206] LMCache INFO: Creating transfer context (device_type=cpu, mode=auto) (worker_transfer.py:551:lmcache.v1.multiprocess.transfer_context.worker_transfer)
[2026-06-23 23:26:52,206] LMCache INFO: Engine KV Format: EngineKVFormat.NL_X_NB_TWO_NH_BS_HS NL x [NB, 2, NH, BS, HS] (detection.py:44:lmcache.v1.gpu_connector.kv_format.detection)
[2026-06-23 23:26:52,208] LMCache INFO: Creating EngineDrivenContextShm (shm_name=lmcache_l1_pool_lmcache_kvcache_sdk_e2e, pool_size=150323855360) (base.py:235:lmcache.v1.multiprocess.transfer_context.base)
[2026-06-23 23:26:52,209] LMCache INFO: Worker non-GPU transfer context registered (instance_id=3403760, mode=SHM) (worker_transfer.py:420:lmcache.v1.multiprocess.transfer_context.worker_transfer)


In [6]:
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=trust_remote_code,
)
source_tokens, target_tokens, cache_tokens = _build_prompts(
    tokenizer,
    min_prompt_tokens=min_prompt_tokens,
    chunk_size=chunk_size,
    fake_prefix_tokens=fake_prefix_tokens,
)

print("== Step 1: source inference stores KV under source token IDs ==")
source_completion = _post_completion(
    vllm_url=vllm_url,
    model_name=model_name,
    prompt=source_tokens,
    max_tokens=max_tokens,
    timeout=timeout,
)

print("== Step 2: retrieve source KV into memory through lmcache.sdk ==")
retrieved_kv = lmc_sdk.retrieve(
    ctx=ctx,
    tokens=source_tokens,
)
if retrieved_kv is None:
    raise RuntimeError("source retrieve missed the expected cached prefix")
retrieved_hit_tokens = int(retrieved_kv.shape[2])
retrieved_hit_chunks = retrieved_hit_tokens // chunk_size
if retrieved_hit_tokens < cache_tokens:
    raise RuntimeError(
        "source retrieve did not return the expected cached prefix: "
        f"{retrieved_hit_tokens} < {cache_tokens}"
    )

target_prefix_tokens = target_tokens[:retrieved_hit_tokens]
source_prefix_tokens = source_tokens[:retrieved_hit_tokens]
if source_prefix_tokens == target_prefix_tokens:
    raise RuntimeError("source and target token prefixes unexpectedly match")

print("== Step 3: store source KV under different target token IDs ==")
store_result = lmc_sdk.store(
    ctx=ctx,
    kv=retrieved_kv,
    tokens=target_prefix_tokens,
)
if not store_result:
    raise RuntimeError(
        f"Failed to store {len(target_prefix_tokens)} target prefix tokens. "
        f"Might be because LMCache already has the KV cache."
    )

print("== Step 4: target inference reuses the remapped token IDs ==")
target_completion = _post_completion(
    vllm_url=vllm_url,
    model_name=model_name,
    prompt=target_tokens,
    max_tokens=max_tokens,
    timeout=timeout,
)
outputs_match = source_completion.text == target_completion.text

evaluation = {
    "vllm_model_name": model_name,
    "cache_tokens": cache_tokens,
    "source_prompt_tokens": len(source_tokens),
    "target_prompt_tokens": len(target_tokens),
    "retrieved_hit_tokens": retrieved_hit_tokens,
    "retrieved_hit_chunks": retrieved_hit_chunks,
    "source_target_same_length": len(source_tokens) == len(target_tokens),
    "source_target_last_hit_tokens_match": source_tokens[-retrieved_hit_tokens:]
    == target_tokens[-retrieved_hit_tokens:],
    "target_prefix_ids_differ_from_source": source_prefix_tokens
    != target_prefix_tokens,
    "outputs_match": outputs_match,
    "store_result": store_result,
    "source_retrieve_after_inference": {
        "shape": tuple(retrieved_kv.shape),
        "dtype": str(retrieved_kv.dtype),
        "hit_tokens": retrieved_hit_tokens,
        "hit_chunks": retrieved_hit_chunks,
    },
    "source_latency_seconds": source_completion.elapsed_seconds,
    "target_latency_seconds": target_completion.elapsed_seconds,
    "source_output_preview": " ".join(source_completion.text.split()[:max_tokens]),
    "target_output_preview": " ".join(target_completion.text.split()[:max_tokens]),
}
print("== Evaluation ==")
print(json.dumps(cast(dict[str, object], evaluation), indent=2, default=str))
if not outputs_match:
    raise RuntimeError("Target output did not match source output after KV remap.")

== Step 1: source inference stores KV under source token IDs ==
== Step 2: retrieve source KV into memory through lmcache.sdk ==
== Step 3: store source KV under different target token IDs ==
== Step 4: target inference reuses the remapped token IDs ==
== Evaluation ==
{
  "vllm_model_name": "Qwen/Qwen3-8B",
  "cache_tokens": 512,
  "source_prompt_tokens": 512,
  "target_prompt_tokens": 512,
  "retrieved_hit_tokens": 512,
  "retrieved_hit_chunks": 2,
  "source_target_same_length": true,
  "source_target_last_hit_tokens_match": false,
  "target_prefix_ids_differ_from_source": true,
  "outputs_match": true,
  "store_result": true,
  "source_retrieve_after_inference": {
    "shape": [
      2,
      36,
      512,
      1024
    ],
    "dtype": "torch.bfloat16",
    "hit_tokens": 512,
    "hit_chunks": 2
  },
  "source_latency_seconds": 1.2143411422148347,
  "target_latency_seconds": 1.1793114291504025,
  "source_output_preview": "Okay, I need to understand what the user is asking for. Th

In [7]:
# Only close the context when done.
lmc_sdk.close(ctx)